In [ ]:
import os
import scanpy as sc
import pandas as pd
import warnings

# Suppress pandas warnings for a cleaner output
warnings.filterwarnings('ignore')

def build_annotated_adata(data_dir, sample_meta_path, clinical_meta_path):
    """
    Load 10x MTX data and integrate clinical metadata into an AnnData object.

    Parameters:
    -----------
    data_dir : str
        Path to the directory containing the 10x matrix files (matrix.mtx, barcodes, features).
    sample_meta_path : str
        Path to the sample.txt.gz metadata file.
    clinical_meta_path : str
        Path to the clinical information.xlsx file.

    Returns:
    --------
    AnnData
        An AnnData object with integrated metadata in the .obs attribute.
    """
    print("[1/4] Loading scRNA-seq matrix data...")
    adata = sc.read_10x_mtx(data_dir, cache=True)
    adata.var_names_make_unique()

    print("[2/4] Integrating sample metadata...")
    sample_df = pd.read_csv(sample_meta_path, sep="\t")
    sample_df = sample_df.set_index("Cell")
    adata.obs = adata.obs.join(sample_df[["Type", "Sample", "S_ID"]], how="left")

    print("[3/4] Processing and standardizing clinical information...")
    clinical_df = pd.read_excel(clinical_meta_path)
    clinical_df = clinical_df[["IDa", "Diagnosis", "Stage"]].rename(columns={"IDa": "Sample"})

    # Create a full diagnosis string (e.g., 'HCC II')
    clinical_df["Diagnosis_full"] = clinical_df["Diagnosis"].astype(str) + " " + clinical_df["Stage"].astype(str)

    # Map clinical stages to numeric values
    stage_map = {"I": "1", "II": "2", "III": "3", "IIIB": "3", "IV": "4"}

    def format_disease_stage(row):
        diag, stage = row["Diagnosis"], row["Stage"]
        prefix = "H" if diag == "HCC" else ("iC" if diag == "iCCA" else "UNK")
        return prefix + stage_map.get(stage, "")

    clinical_df["Disease"] = clinical_df.apply(format_disease_stage, axis=1)

    print("[4/4] Synchronizing IDs and merging clinical data into AnnData...")
    # Dictionary to map fragmented IDs (a, b, c) back to their root IDs (*)
    sample_to_ida = {
    "H49a": "H49*",
    "H49b": "H49*",

    "C46a": "C46*",
    "C46b": "C46*",

    "H73a": "H73*",
    "H73b": "H73*",

    "H34a": "H34*",
    "H34b": "H34*",
    "H34c": "H34*",

    "H58a": "H58*",
    "H58b": "H58*",
    "H58c": "H58*",

    "H68a": "H68*",
    "H68b": "H68*",

    "C26a": "C26*",
    "C26b": "C26*"
    }

    adata.obs["Sample_clinical"] = adata.obs["Sample"].map(sample_to_ida).fillna(adata.obs["Sample"])

    # Merge clinical data
    adata.obs = adata.obs.join(
        clinical_df.set_index("Sample")[["Diagnosis_full", "Disease"]],
        on="Sample_clinical",
        how="left"
    )

    # Drop intermediate columns to keep the object clean
    adata.obs.drop(columns=["Sample_clinical"], inplace=True)

    print(f"[*] Complete! Data shape: {adata.n_obs} cells, {adata.n_vars} genes.")
    return adata

# ==========================================
# EXECUTION BLOCK
# ==========================================
# Define relative paths (Suitable for GitHub repository structure)
DATA_DIR = "./data/GSE151530/"
SAMPLE_META = "./data/GSE151530/sample.txt.gz"
CLINICAL_META = "./data/clinical_information.xlsx"

# Execute the function
adata = build_annotated_adata(DATA_DIR, SAMPLE_META, CLINICAL_META)

# Display the first 5 rows of metadata to verify
print(adata.obs.head())